In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os 
import numpy as np
import scipy as sci
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as pl

import scanpy as sc
import scirpy as ir
import anndata as ann
import awkward as ak
import muon as mu

from scipy.sparse import csr_matrix
from matplotlib import rcParams
from matplotlib import colors

# Set scanpy settings
sc.settings.verbosity = 3  # verbosity level
sc.settings.set_figure_params(dpi=80, facecolor='white')


e:\Anaconda\envs\mvTCR\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\Anaconda\envs\mvTCR\lib\site-packages\airr\schema.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream


In [2]:
# Define file paths
reference_path = r"E:\Python code\Machine learning\JupyterNote\Bio_CRC\Data\Lee_EAE\mvTCR_processed\p1_preprocessed_with_metadata.h5ad"
query_path = r"E:\Python code\Machine learning\JupyterNote\Bio_CRC\Data\EAE\GSE188320\10x_processed_all_genes.h5ad"


In [118]:
# Load reference data
print("Loading reference data...")
adata_ref = sc.read_h5ad(reference_path)
adata_ref


Loading reference data...


AnnData object with n_obs × n_vars = 51500 × 2500
    obs: 'mouse_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_counts', 'log_counts', 'n_genes', 'mt_fraction', 'doublet_score', 'doublet', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonotype', 'clonotype_size', 'alpha_len', 'beta_len', 'has_binding', 'set', 'VJ_1_cdr3_aa', 'VJ_1_v_call', 'VJ_1_j_call', 'VDJ_1_cdr3_aa', 'VDJ_1_v_call', 'VDJ_1_j_call', 'cloned', 'in_two_tissue', 'modified_cell_type', 'state'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'aa_to_id', 'chain_indices', 'clone_id', 'clonotype', 'hvg', 'ir_dist_nt_identity', 'log1p', 'mouse_id_colors', 'mouse_id_enc'
    obsm: 'airr', 'alpha_seq', 'beta_seq', 'chain_indices', 'mouse_id_ohe'

In [119]:
adata_query = sc.read_h5ad(query_path)
adata_query

AnnData object with n_obs × n_vars = 28652 × 14358
    obs: 'batch', 'condition', 'mouse_id', 'assignment_demuxem', 'n_counts', 'log_counts', 'n_genes', 'mt_fraction', 'doublet_score', 'doublet', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonotype', 'clonotype_size', 'alpha_len', 'beta_len'
    var: 'n_cells'
    uns: 'aa_to_id', 'chain_indices', 'clone_id', 'clonotype', 'ir_dist_nt_identity', 'log1p', 'mouse_id_colors'
    obsm: 'airr', 'alpha_seq', 'beta_seq', 'chain_indices'

In [120]:
import anndata as ad
def merge_anndata_to_base(anndata1, anndata2):
    c_gene = anndata2.var_names.intersection(anndata1.var_names)

    missing_genes = anndata1.var_names.difference(anndata2.var_names)

    # Pad missing genes with zeros (assumes dense arrays, can be adapted for sparse)
    if len(missing_genes) > 0:
        import pandas as pd
        import scipy.sparse

        shape = (anndata2.n_obs, len(missing_genes))
        X_pad = np.zeros(shape, dtype=anndata2.X.dtype)
        X_pad = scipy.sparse.csr_matrix(X_pad) if isinstance(anndata2.X, scipy.sparse.spmatrix) else X_pad

        # Create dummy .var for missing genes
        var_pad = pd.DataFrame(index=missing_genes)

        # Create a temporary AnnData with padded genes
        adata_pad = ad.AnnData(X=X_pad, obs=anndata2.obs.copy(), var=var_pad)

        # Add missing genes and reorder to match anndata1
        anndata2_full = ad.concat([anndata2, adata_pad], axis=1, join="outer", merge="first" )
        anndata2_full = anndata2_full[:, anndata1.var_names]

    else:
        anndata2_full = anndata2[:, anndata1.var_names]
    return anndata2_full


In [121]:
# Add dataset labels to obs
query_merged = merge_anndata_to_base(adata_ref, adata_query)

assert set(query_merged.var_names) == set(adata_ref.var_names), "Gene alignment failed!"
print("✓ Gene alignment successful!")

query_merged


✓ Gene alignment successful!


View of AnnData object with n_obs × n_vars = 28652 × 2500
    obs: 'batch', 'condition', 'mouse_id', 'assignment_demuxem', 'n_counts', 'log_counts', 'n_genes', 'mt_fraction', 'doublet_score', 'doublet', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonotype', 'clonotype_size', 'alpha_len', 'beta_len'
    var: 'n_cells'
    obsm: 'airr', 'alpha_seq', 'beta_seq', 'chain_indices'

In [ ]:
# Rename 'assignment_demuxem' to 'tissue' in query_merged.obs

query_merged.obs = query_merged.obs.rename(columns={'assignment_demuxem': 'tissue'})

# Rename 'SP' to 'SPL' and 'CN' to 'CNS' in adata_ref.obs['tissue']
adata_ref.obs['tissue'] = adata_ref.obs['tissue'].replace({'SP': 'SPL', 'CN': 'CNS'})

In [ ]:
query_merged.obs['tissue']

AAACCTGAGGCTCAGA-1-b6m1    MLN
AAACCTGCAATAGCAA-1-b6m1    SPL
AAACCTGCAGTCTTCC-1-b6m1    COL
AAACCTGCATTCTTAC-1-b6m1    SPL
AAACCTGGTACATCCA-1-b6m1    SPL
                          ... 
TTTGGTTTCGCTTAGA-1-b9m2     PP
TTTGTCAAGTATCTCG-1-b9m2    COL
TTTGTCAGTCGCATCG-1-b9m2    SPL
TTTGTCAGTTTGTGTG-1-b9m2     PP
TTTGTCATCCTGCTTG-1-b9m2    NaN
Name: tissue, Length: 28652, dtype: category
Categories (7, object): ['CNS', 'COL', 'DLN', 'MLN', 'PP', 'SI', 'SPL']

In [124]:
adata_ref.obs['set'] = 'train'
query_merged.obs['set'] = 'test'

In [125]:
# Find common obs columns between query_merged and adata_ref
common_obs = set(query_merged.obs.columns) & set(adata_ref.obs.columns)

# For each common obs column, match dtype to adata_ref if different
for col in common_obs:
    ref_dtype = adata_ref.obs[col].dtype
    query_dtype = query_merged.obs[col].dtype
    if ref_dtype != query_dtype:
        try:
            # Try to cast query_merged.obs[col] to ref_dtype
            query_merged.obs[col] = query_merged.obs[col].astype(ref_dtype)
        except Exception as e:
            print(f"Could not convert column '{col}' from {query_dtype} to {ref_dtype}: {e}")

In [129]:
ad_merged = ad.concat([adata_ref, query_merged], axis=0, join="outer", merge='first')
ad_merged

AnnData object with n_obs × n_vars = 80152 × 2500
    obs: 'mouse_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_counts', 'log_counts', 'n_genes', 'mt_fraction', 'doublet_score', 'doublet', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonotype', 'clonotype_size', 'alpha_len', 'beta_len', 'has_binding', 'set', 'VJ_1_cdr3_aa', 'VJ_1_v_call', 'VJ_1_j_call', 'VDJ_1_cdr3_aa', 'VDJ_1_v_call', 'VDJ_1_j_call', 'cloned', 'in_two_tissue', 'modified_cell_type', 'state', 'batch', 'condition'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    obsm: 'airr', 'alpha_seq', 'beta_seq', 'chain_indices', 'mouse_id_ohe'

In [ ]:
ad_merged.obs['tissue']

AAACCAAAGGGGAGCT-1_0516_CNS    CNS
AAACCAGCACGTAAAG-1_0516_CNS    CNS
AAACCATTCCTCCGGT-1_0516_CNS    CNS
AAACCCATCAGTATCG-1_0516_CNS    CNS
AAACCCCAGCCTAAGC-1_0516_CNS    CNS
                              ... 
TTTGGTTTCGCTTAGA-1-b9m2        NaN
TTTGTCAAGTATCTCG-1-b9m2        NaN
TTTGTCAGTCGCATCG-1-b9m2        SPL
TTTGTCAGTTTGTGTG-1-b9m2        NaN
TTTGTCATCCTGCTTG-1-b9m2        NaN
Name: tissue, Length: 80152, dtype: category
Categories (2, object): ['CNS', 'SPL']

In [ ]:
ad_merged.write('merged_EAE.h5ad')